In [16]:
import gradio as gr
from __future__ import annotations
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from io import BytesIO
from PIL import Image

# ----------------------------
# Object Configuration
# ----------------------------

@dataclass(frozen=True)
class Obj:
    name: str
    color: str
    shape: str  # "square", "triangle", "block", "circle", "rectangle", "pentagon"
    size: str  # "small", "medium", "large"
    weight: int  # Weight in tons

    @property
    def description(self) -> str:
        """Returns a natural language description, e.g., 'small red square'."""
        return f"{self.size} {self.color} {self.shape} ({self.weight}t)"

# ----------------------------
# World Environment State
# ----------------------------

@dataclass
class World:
    objects: Dict[str, Obj]
    zone_assignment: Dict[str, str]     # Maps obj_name -> "shelf" (Area A), "docks" (Area B), or "ship"
    whoIsUnder: Dict[str, Optional[str]] # Mapping: top_obj -> support_obj (None means base floor of its zone)
    rightNeighbors: Dict[str, List[str]] # Horizontal tracking chain within a zone floor
    holding: Optional[str] = None
    last_zone: Optional[str] = None     # TRACKER: Remembers the zone an object was in before pickup
    last_support: Optional[str] = None  # TRACKER: Remembers the specific object it was stacked on

    def get_zone_weight(self, zone: str) -> int:
        """Calculates the total weight of all objects currently inside a given zone."""
        return sum(obj.weight for name, obj in self.objects.items() if self.zone_assignment.get(name) == zone and name != self.holding)

    def is_clear(self, obj_name: str) -> bool:
        """True if nothing is stacked directly on top of obj_name."""
        for support in self.whoIsUnder.values():
            if support == obj_name:
                return False
        return True

    def top_of(self, obj_name: str) -> Optional[str]:
        """Return object that is on top of obj_name (if any)."""
        for x, support in self.whoIsUnder.items():
            if support == obj_name:
                return x
        return None

    def describe(self) -> str:
        lines = []
        zone_display = {"shelf": "Loading Area A", "docks": "Loading Area B", "ship": "Cargo Ship"}
        for x in sorted(self.objects):
            support = self.whoIsUnder.get(x, None)
            zone = zone_display.get(self.zone_assignment.get(x, "docks"))
            if support is None and x != self.holding:
                lines.append(f"{self.objects[x].description} ({x}) is on the floor of {zone}")
            elif x != self.holding:
                lines.append(f"{self.objects[x].description} ({x}) is on {self.objects[support].description} ({support}) (in {zone})")
        lines.append(f"Holding: {self.holding}")
        lines.append(f"Cargo Ship Weight: {self.get_zone_weight('ship')}/50 Tons")
        return "\n".join(lines)

    def pickup(self, x: str) -> None:
        """Picks up an object if it is clear and the crane is empty."""
        if self.holding is not None and self.holding != x:
            raise RuntimeError(f"The crane is already holding {self.objects[self.holding].description}. You must put it down first!")
        
        if not self.is_clear(x):
            blocker = self.top_of(x)
            raise RuntimeError(f"Cannot pick up the {self.objects[x].description}: The {self.objects[blocker].description} is on top of it!")
            
        rights_of_x = self.rightNeighbors.get(x, [])
        for name, neighbors in self.rightNeighbors.items():
            if neighbors and x in neighbors:
                neighbors.remove(x)
                for r in rights_of_x:
                    if r not in neighbors:
                        neighbors.append(r)
        
        # --- TRACKING LOGIC ---
        self.last_zone = self.zone_assignment.get(x, "docks")
        self.last_support = self.whoIsUnder.get(x, None) # Remember what it was stacked on (if anything)
        
        self.holding = x
        self.whoIsUnder[x] = None
        self.rightNeighbors[x] = []

    def put_on(self, x: str, y: str) -> None:
        """Places held object x onto object y with a maximum stack height limit of 5."""
        if x == y:
            raise RuntimeError(f"Cannot place the {self.objects[x].description} on top of itself!")       
        if self.holding != x:
            raise RuntimeError(f"Not holding {self.objects[x].description}.")
        if not self.is_clear(y):
            raise RuntimeError(f"Cannot place on {self.objects[y].description}: It is not clear.")
        
        target_zone = self.zone_assignment[y]
        if target_zone == "ship":
            current_ship_weight = self.get_zone_weight("ship")
            if current_ship_weight + self.objects[x].weight > 50:
                raise RuntimeError(f"Placement denied! Adding {self.objects[x].description} ({self.objects[x].weight}t) would exceed the Cargo Ship's maximum limit of 50 tons (Current: {current_ship_weight}t).")

        stack_height = 1
        current = y
        while self.whoIsUnder.get(current) is not None:
            stack_height += 1
            current = self.whoIsUnder[current]
        
        if stack_height + 1 > 5:
            raise RuntimeError("Stack height limit is 5, please choose another stack")

        self.whoIsUnder[x] = y
        self.zone_assignment[x] = target_zone
        self.holding = None
        self.last_zone = None
        self.last_support = None
    
    def put_down(self, x: str, target_zone: str, target_support: Optional[str] = None) -> None:
        """Places held object down. If a target_support is specified and clear, stacks it there. Otherwise, floors it."""
        if self.holding != x:
            raise RuntimeError(f"Not holding {self.objects[x].description}.")
        
        # If it was previously on an object, and that object is still clear, route it straight to put_on
        if target_support is not None:
            if target_support in self.objects and self.is_clear(target_support):
                self.put_on(x, target_support)
                return
        
        if target_zone == "ship":
            current_ship_weight = self.get_zone_weight("ship")
            if current_ship_weight + self.objects[x].weight > 50:
                raise RuntimeError(f"Placing {self.objects[x].description} would exceed the Cargo Ship's maximum limit of 50 tons (Current: {current_ship_weight}t)!")

        # Fallback to standard baseline floor behavior if there was no support or if the support is blocked
        on_floor = [
            name for name, support in self.whoIsUnder.items() 
            if support is None and name != self.holding and self.zone_assignment.get(name) == target_zone
        ]
        
        zone_display_names = {"shelf": "Loading Area A", "docks": "Loading Area B", "ship": "Cargo Ship"}
        limit = 5 if target_zone == "ship" else 7
        
        if len(on_floor) >= limit:
            raise RuntimeError(
                f"{zone_display_names[target_zone]} floor space is full, "
                f"consider stacking objects on top of each other with the put on command"
            )
        
        rightmost_obj = None
        for obj_name in on_floor:
            if not self.rightNeighbors.get(obj_name, []):
                rightmost_obj = obj_name
                break
                
        if rightmost_obj:
            self.rightNeighbors[rightmost_obj] = [x]
            
        self.rightNeighbors[x] = []
        self.whoIsUnder[x] = None
        self.zone_assignment[x] = target_zone
        self.holding = None
        self.last_zone = None
        self.last_support = None

In [17]:
# Initialize default world state containing all 13 original objects unstacked with varied weights
def create_initial_world():
    return World(
        objects={
            # Small sizes: ~2-4 tons
            "c2": Obj("c2", "green", "triangle", "small", 2),
            "b3": Obj("b3", "orange", "square", "small", 3),
            "c4": Obj("c4", "yellow", "square", "small", 4),
            "l2": Obj("l2", "blue", "circle", "small", 3),
            # Medium sizes: ~5-8 tons
            "c1": Obj("c1", "red", "square", "medium", 6),
            "b2": Obj("b2", "blue", "square", "medium", 5),
            "l1": Obj("l1", "orange", "circle", "medium", 7),
            "s1": Obj("s1", "yellow", "pentagon", "medium", 8),
            "c3": Obj("c3", "pink", "triangle", "medium", 6),
            # Large sizes: ~10-14 tons
            "p1": Obj("p1", "blue", "triangle", "large", 12),
            "r1": Obj("r1", "pink", "rectangle", "large", 10),
            "s2": Obj("s2", "red", "pentagon", "large", 14),
            "l3": Obj("l3", "green", "circle", "large", 11),
        },
        whoIsUnder={
            "c1": None, "b2": None, "p1": None, "c2": None, "l1": None,
            "r1": None, "s1": None, "s2": None, "l2": None, "l3": None,
            "b3": None, "c3": None, "c4": None
        },
        rightNeighbors={
            "c1": ["b2"], "b2": ["p1"], "p1": ["c2"], "c2": [],
            "l1": ["r1"], "r1": ["s1"], "s1": ["s2"], "s2": [],
            "l2": ["l3"], "l3": ["b3"], "b3": ["c3"], "c3": ["c4"], "c4": []
        },
        zone_assignment={
            "c1": "shelf", "b2": "shelf", "p1": "shelf", "c2": "shelf",
            "l1": "docks", "r1": "docks", "s1": "docks", "s2": "docks",
            "l2": "ship", "l3": "ship", "b3": "ship", "c3": "ship", "c4": "ship"
        }
    )

world_state = create_initial_world()

In [18]:
# ----------------------------------------
# ACTION EXECUTION PLANS
# ----------------------------------------

def plan_pickup(world: World, x: str) -> List[Tuple[str, str, Optional[str], str]]:
    if world.holding is not None and world.holding != x:
        raise RuntimeError(f"The crane is already holding {world.objects[world.holding].description}. You must put it down first!")
    
    if not world.is_clear(x):
        blocker = world.top_of(x)
        raise RuntimeError(f"Cannot move or pick up the {world.objects[x].description} because the {world.objects[blocker].description} is on top of it!")
        
    return [("pickup", x, None, "")]

def plan_put(world: World, x: str, y: Optional[str], target_zone: str) -> List[Tuple[str, str, Optional[str], str]]:
    if y is not None and not world.is_clear(y):
        blocker = world.top_of(y)
        raise RuntimeError(f"Cannot place onto the {world.objects[y].description} because the {world.objects[blocker].description} is blocking it!")
            
    if world.holding is not None and world.holding != x:
        raise RuntimeError(f"The crane is already holding {world.objects[world.holding].description}. You must put it down before managing other containers!")

    plan = plan_pickup(world, x)
    plan.append(("put_on", x, y, target_zone))
    return plan

def execute_plan(world: World, plan: List[Tuple[str, str, Optional[str], str]]) -> None:
    for action, obj, target, zone in plan:
        if action == "pickup":
            world.pickup(obj)
        elif action == "put_on":
            if target is None:
                world.put_down(obj, target_zone=zone)
            else:
                world.put_on(obj, target)


In [ ]:
# ----------------------------------------
# SEMANTIC COMMAND PARSING
# ----------------------------------------

import numpy as np
import joblib

model = joblib.load("stage2/intent_model.pkl")
def predict_intent(text, threshold=0.50):
    probs = model.predict_proba([text])[0]
    classes = model.classes_

    best_idx = np.argmax(probs)
    best_prob = probs[best_idx]
    best_label = classes[best_idx]

    if best_prob < threshold:
        return "UNKNOWN", best_prob
    else:
        return best_label, best_prob


COLORS = {"red", "green", "blue", "yellow", "pink", "orange"}
SHAPES = {"triangle", "square", "rectangle", "circle", "pentagon", "block"}
SIZES = {"small", "medium", "large"}

def parse_command(text: str) -> dict:
    t = text.lower().replace("?", "").strip()

    if "next to" in t or "beside" in t or "left of" in t or "right of" in t:
        raise ValueError("Horizontal placement commands ('next to', 'beside') are not supported. Please use 'put on' or move items directly to a zone floor.")

    tokens = t.split()

    # View Details structural identifier check
    if t.startswith("view details") or t.startswith("inspect") or t.startswith("show details"):
        clean_tokens = [w for w in tokens if w not in ["view", "details", "inspect", "show", "of", "the", "a", "an"]]
        color = next((w for w in clean_tokens if w in COLORS), None)
        shape = next((w for w in clean_tokens if w in SHAPES), "block")
        size = next((w for w in clean_tokens if w in SIZES), None)
        name = next((w for w in clean_tokens if w in world_state.objects), None)
        return {"intent": "VIEW_DETAILS", "ref": {"color": color, "shape": shape, "size": size, "name": name}}


    intent, conf = predict_intent(t)
    if intent == "UNKNOWN":
        raise ValueError("I couldn't understand the command confidently.")


    # map ML → your internal system
    if intent == "pick_up":
        color = next((w for w in tokens if w in COLORS), None)
        shape = next((w for w in tokens if w in SHAPES), "block")
        size = next((w for w in tokens if w in SIZES), "medium")
        name = next((w for w in tokens if w in world_state.objects), None)
        return {"intent": "PICKUP", "ref": {"color": color, "shape": shape, "size": size, "name": name}}

    if intent == "put_down":
        split_word = None
        for prep in ["on", "in", "to", "into", "down"]:
            if prep in tokens:
                split_word = prep
                break

        if split_word:
            split_i = tokens.index(split_word)
            left = tokens[1:split_i]
            right = tokens[split_i+1:]
            
            left = [w for w in left if w not in ["the", "a", "an"]]
            right = [w for w in right if w not in ["the", "a", "an", "loading", "area", "cargo"]]

            # Explicit object "put down" pattern: e.g. "put down the red square"
            if split_word == "down" and not right:
                color_x = next((w for w in left if w in COLORS), None)
                shape_x = next((w for w in left if w in SHAPES), "block")
                name_x = next((w for w in left if w in world_state.objects), None)
                return {"intent": "PUT_DOWN_EXPLICIT", "x": {"color": color_x, "shape": shape_x, "name": name_x}}

            if target_zone is not None or any(w in ["ship", "shelf", "dock", "docks", "vessel", "a", "b"] for w in right):
                color_x = next((w for w in left if w in COLORS), None)
                shape_x = next((w for w in left if w in SHAPES), "block")
                name_x = next((w for w in left if w in world_state.objects), None)
                return {"intent": "PUT_DOWN", "x": {"color": color_x, "shape": shape_x, "name": name_x}, "zone": target_zone}
        
        else:
            limit_i = len(tokens)
            for stop_word in ["to", "in", "into", "area", "ship"]:
                if stop_word in tokens:
                    limit_i = min(limit_i, tokens.index(stop_word))
            left = tokens[1:limit_i]
            left = [w for w in left if w not in ["the", "a", "an"]]
            color_x = next((w for w in left if w in COLORS), None)
            shape_x = next((w for w in left if w in SHAPES), "block")
            name_x = next((w for w in left if w in world_state.objects), None)
            return {"intent": "PUT_DOWN", "x": {"color": color_x, "shape": shape_x, "name": name_x}, "zone": target_zone}
        #return {"intent": "PUT_DOWN_ANONYMOUS", "confidence": conf}
    
    if intent == "place":

    # -------------------------
    # 1. ZONE DETECTION FIRST
    # -------------------------
    target_zone = None

    if "area a" in t:
        target_zone = "area_a"
    elif "area b" in t:
        target_zone = "area_b"
    elif "ship" in t:
        target_zone = "ship"

    if target_zone is not None:
        # PUT INTO LOCATION
        color_x = next((w for w in tokens if w in COLORS), None)
        shape_x = next((w for w in tokens if w in SHAPES), "block")
        name_x = next((w for w in tokens if w in world_state.objects), None)

        return {
            "intent": "PUT_IN",
            "x": {"color": color_x, "shape": shape_x, "name": name_x},
            "zone": target_zone
        }

    # -------------------------
    # 2. OBJECT-OBJECT RELATION
    # -------------------------
    split_word = None
    for prep in ["on", "onto", "over", "above", "stack"]:
        if prep in tokens:
            split_word = prep
            break

    if split_word:

        split_i = tokens.index(split_word)
        left = tokens[:split_i]
        right = tokens[split_i + 1:]

        left = [w for w in left if w not in ["put", "place", "stack", "the", "a", "an"]]
        right = [w for w in right if w not in ["the", "a", "an"]]

        color_x = next((w for w in left if w in COLORS), None)
        shape_x = next((w for w in left if w in SHAPES), "block")
        name_x = next((w for w in left if w in world_state.objects), None)

        color_y = next((w for w in right if w in COLORS), None)
        shape_y = next((w for w in right if w in SHAPES), "block")
        name_y = next((w for w in right if w in world_state.objects), None)

        return {
            "intent": "PUT_ON",
            "x": {"color": color_x, "shape": shape_x, "name": name_x},
            "y": {"color": color_y, "shape": shape_y, "name": name_y}
        }
        
    raise ValueError("I'm not sure what you wanted me to do.")

def resolve_ref(world: World, color: Optional[str], shape: Optional[str], name: Optional[str] = None) -> List[str]:
    if name and name in world.objects:
        return [name]
    matches = []
    for o_name, obj in world.objects.items():
        if color is not None and obj.color != color:
            continue
        if shape is not None and shape != "block" and obj.shape != shape:
            continue
        matches.append(o_name)
    return matches

def choose_unique(matches: List[str], what: str) -> str:
    if not matches:
        raise ValueError(f"I can't find any {what} matching that description.")
    if len(matches) > 1:
        raise ValueError(f"I see multiple options for {what}: {matches}. Please be more specific!")
    return matches[0]

def interpret_and_act(world: World, utterance: str) -> str:
    parsed = parse_command(utterance)
    zone_names = {"shelf": "Loading Area A", "docks": "Loading Area B", "ship": "Cargo Ship"}
    
    # 0. View Details Profile inspection lookup
    if parsed["intent"] == "VIEW_DETAILS":
        ref = parsed["ref"]
        matches = resolve_ref(world, ref["color"], ref["shape"], ref["name"])
        desc_str = f"{ref['size'] or ''} {ref['color'] or ''} {ref['shape']}".strip()
        x = choose_unique(matches, ref["name"] or desc_str)
        obj = world.objects[x]
        return (
            f"🔍 **Object Profile Details:**\n"
            f"• **ID (Name):** `{x}`\n"
            f"• **Size:** {obj.size.capitalize()}\n"
            f"• **Color:** {obj.color.capitalize()}\n"
            f"• **Shape:** {obj.shape.capitalize()}\n"
            f"• **Weight Limit Metric:** **{obj.weight} Tons**"
        )
    
    elif parsed["intent"] == "PUT_DOWN":
        mx = resolve_ref(world, parsed["x"]["color"], parsed["x"]["shape"], parsed["x"]["name"])
        x = choose_unique(mx, parsed["x"]["name"] or f"{parsed['x']['color'] or ''} {parsed['x']['shape']}".strip())
        
        # If no explicit zone target is listed via parser, match context
        target_zone = parsed["zone"] if parsed["zone"] else (world.last_zone if world.holding == x else "docks")
        support_item = world.last_support if world.holding == x else None
        
        if world.holding == x:
            if support_item and not world.is_clear(support_item):
                support_item = None # Clear target back to floor if old target is now covered
            world.put_down(x, target_zone, target_support=support_item)
        else:
            plan = plan_pickup(world, x)
            execute_plan(world, plan)
            # Re-read tracking stats post-pickup step
            support_item = world.last_support
            if support_item and not world.is_clear(support_item):
                support_item = None
            world.put_down(x, target_zone, target_support=support_item)
            
        if support_item and world.whoIsUnder.get(x) == support_item:
            return f"OK. Moved {world.objects[x].description} back on top of {world.objects[support_item].description}."
        return f"OK. Moved {world.objects[x].description} into {zone_names[target_zone]}."
    
    elif parsed["intent"] == "PUT_DOWN_EXPLICIT":
        mx = resolve_ref(world, parsed["x"]["color"], parsed["x"]["shape"], parsed["x"]["name"])
        x = choose_unique(mx, parsed["x"]["name"] or f"{parsed['x']['color'] or ''} {parsed['x']['shape']}".strip())
        
        if world.holding != x:
            if world.holding is not None:
                raise RuntimeError(f"The crane is not holding that object. It is holding the {world.objects[world.holding].description} instead.")
            else:
                raise RuntimeError(f"The crane is not holding the {world.objects[x].description}. You must pick it up first.")
        
        destination_zone = world.last_zone if world.last_zone else "docks"
        support_item = world.last_support
        
        if support_item and not world.is_clear(support_item):
            raise RuntimeError(f"Cannot return to original position: Object {world.objects[support_item].description} ({support_item}) now has something else stacked on it!")
            
        world.put_down(x, target_zone=destination_zone, target_support=support_item)
        if support_item and world.whoIsUnder.get(x) == support_item:
            return f"OK. Put down {world.objects[x].description} back on top of {world.objects[support_item].description}."
        return f"OK. Put down {world.objects[x].description} back into {zone_names[destination_zone]}."

    elif parsed["intent"] == "PICKUP":
        ref = parsed["ref"]
        matches = resolve_ref(world, ref["color"], ref["shape"], ref["name"])
        x = choose_unique(matches, ref["name"] or f"{ref['color'] or ''} {ref['shape']}".strip())
        plan = plan_pickup(world, x)
        execute_plan(world, plan)
        return f"OK. Picked up {world.objects[x].description}."
        
    elif parsed["intent"] == "PUT_ON":
        mx = resolve_ref(world, parsed["x"]["color"], parsed["x"]["shape"], parsed["x"]["name"])
        my = resolve_ref(world, parsed["y"]["color"], parsed["y"]["shape"], parsed["y"]["name"])
        x = choose_unique(mx, parsed["x"]["name"] or f"{parsed['x']['color'] or ''} {parsed['x']['shape']}".strip())
        y = choose_unique(my, parsed["y"]["name"] or f"{parsed['y']['color'] or ''} {parsed['y']['shape']}".strip())
        plan = plan_put(world, x, y, target_zone=world.zone_assignment[y])
        execute_plan(world, plan)
        return f"OK. Moved {world.objects[x].description} on top of {world.objects[y].description}."
        
    elif parsed["intent"] == "PUT_IN":
        return f"Oops not trained for put in."

    else:
        raise ValueError("Unsupported intent.")


In [20]:

# ----------------------------------------
# MULTI-LEVEL SPATIAL VISUALIZATION
# ----------------------------------------

size_map = {"small": 0.3, "medium": 0.5, "large": 0.8}

def get_width(obj):
    return size_map[obj.size]

def get_height(obj):
    s = size_map[obj.size]
    if obj.shape == "triangle": return s * 1.2
    if obj.shape == "rectangle": return s / 1.5
    if obj.shape == "pentagon": return s * 0.6
    return s

def draw_object(obj, center_x, y_pos):
    w = get_width(obj)
    h = get_height(obj)
    color = "#FF69B4" if obj.color == "pink" else obj.color
    left = center_x - w / 2
    if obj.shape == "triangle":
        points = [[left, y_pos], [left + w, y_pos], [center_x, y_pos + h]]
        return patches.Polygon(points, closed=True, edgecolor='black', facecolor=color, linewidth=2)
    elif obj.shape == "circle":
        r = w / 2
        return patches.Circle((center_x, y_pos + r), r, edgecolor='black', facecolor=color, linewidth=2)
    elif obj.shape == "pentagon":
        points = [
            [center_x, y_pos + h], [left + w * 0.9, y_pos + h * 0.7],
            [left + w * 0.7, y_pos], [left + w * 0.3, y_pos],
            [left + w * 0.1, y_pos + h * 0.7]
        ]
        return patches.Polygon(points, closed=True, edgecolor='black', facecolor=color, linewidth=2)
    return patches.Rectangle((left, y_pos), w, h, edgecolor='black', facecolor=color, linewidth=2)

def visualize_world(world) -> Image.Image:
    on_shelf = [n for n, s in world.whoIsUnder.items() if s is None and n != world.holding and world.zone_assignment.get(n) == "shelf"]
    on_docks = [n for n, s in world.whoIsUnder.items() if s is None and n != world.holding and world.zone_assignment.get(n) == "docks"]
    on_ship = [n for n, s in world.whoIsUnder.items() if s is None and n != world.holding and world.zone_assignment.get(n) == "ship"]
    
    all_rights = set()
    for neighbors in world.rightNeighbors.values():
        all_rights.update(neighbors)

    def arrange_zone(base_elements):
        leftmosts = [o for o in base_elements if o not in all_rights]
        ordered = []
        visited = set()
        
        def traverse(curr):
            if curr in visited or curr not in base_elements: return
            visited.add(curr)
            ordered.append(curr)
            for nb in world.rightNeighbors.get(curr, []):
                traverse(nb)
                
        for start in sorted(leftmosts):
            traverse(start)
            
        zone_stacks = {}
        for base_obj in ordered:
            stack = [base_obj]
            curr = base_obj
            while len(stack) <= len(world.objects):
                top = world.top_of(curr)
                if top is None or top in stack: break
                stack.append(top)
                curr = top
            zone_stacks[base_obj] = stack
        return zone_stacks

    shelf_stacks = arrange_zone(on_shelf)
    docks_stacks = arrange_zone(on_docks)
    ship_stacks = arrange_zone(on_ship)

    fig, ax = plt.subplots(figsize=(14, 8))
    ax.set_xlim(-0.5, 14)
    ax.set_ylim(-0.5, 8)
    ax.set_aspect('equal')
    ax.axis('off')

    # LEVEL 1 GROUND BASE: LOADING AREA B
    ax.hlines(y=0, xmin=0, xmax=7.5, color='dimgray', linewidth=4, alpha=0.8)
    ax.text(3.75, -0.4, "LOADING AREA B", ha='center', fontsize=11, fontweight='bold', color='dimgray')

    current_x = 0.5
    padding = 0.5
    for base_obj, stack in docks_stacks.items():
        max_w = max(get_width(world.objects[name]) for name in stack)
        center_x = current_x + (max_w / 2)
        y_pos = 0
        for obj_name in stack:
            obj = world.objects[obj_name]
            ax.add_patch(draw_object(obj, center_x, y_pos))
            # Dynamic text color adjustment
            txt_col = "white" if obj.color in ["blue", "red", "green"] else "black"
            ax.text(center_x, y_pos + get_height(obj)/2, f"{obj_name}\n({obj.weight}t)", ha='center', va='center', fontsize=8, fontweight='bold', color=txt_col)
            y_pos += get_height(obj)
        current_x += max_w + padding

    # LEVEL 2 ELEVATED SHELF: LOADING AREA A
    shelf_y = 4.0
    ax.hlines(y=shelf_y, xmin=0, xmax=7.5, color='goldenrod', linewidth=4, alpha=0.9)
    ax.text(3.75, shelf_y - 0.4, "LOADING AREA A", ha='center', fontsize=11, fontweight='bold', color='darkgoldenrod')

    current_shelf_x = 0.5
    for base_obj, stack in shelf_stacks.items():
        max_w = max(get_width(world.objects[name]) for name in stack)
        center_x = current_shelf_x + (max_w / 2)
        y_pos = shelf_y
        for obj_name in stack:
            obj = world.objects[obj_name]
            ax.add_patch(draw_object(obj, center_x, y_pos))
            # Dynamic text color adjustment
            txt_col = "white" if obj.color in ["blue", "red", "green"] else "black"
            ax.text(center_x, y_pos + get_height(obj)/2, f"{obj_name}\n({obj.weight}t)", ha='center', va='center', fontsize=8, fontweight='bold', color=txt_col)
            y_pos += get_height(obj)
        current_shelf_x += max_w + padding

    # ZONE 3 RIGHT SIDE: CARGO SHIP
    ship_left, ship_right, ship_top = 8.5, 13.5, 4.5
    ship_frame = patches.Polygon(
        [[ship_left, ship_top], [ship_left, 0], [ship_right, 0], [ship_right, ship_top]],
        closed=False, edgecolor='darkblue', facecolor='none', linewidth=4
    )
    ax.add_patch(ship_frame)
    
    ship_weight = world.get_zone_weight("ship")
    weight_color = "red" if ship_weight > 40 else "darkblue"
    ax.text((ship_left + ship_right)/2, ship_top + 0.3, f"WEIGHT LOAD: {ship_weight} / 50 TONS", ha='center', fontsize=11, fontweight='bold', color=weight_color)
    ax.text((ship_left + ship_right)/2, -0.4, "CARGO SHIP", ha='center', fontsize=11, fontweight='bold', color='darkblue')

    current_ship_x = ship_left + 0.4
    for base_obj, stack in ship_stacks.items():
        max_w = max(get_width(world.objects[name]) for name in stack)
        center_x = current_ship_x + (max_w / 2)
        y_pos = 0
        for obj_name in stack:
            obj = world.objects[obj_name]
            ax.add_patch(draw_object(obj, center_x, y_pos))
            # Dynamic text color adjustment
            txt_col = "white" if obj.color in ["blue", "red", "green"] else "black"
            ax.text(center_x, y_pos + get_height(obj)/2, f"{obj_name}\n({obj.weight}t)", ha='center', va='center', fontsize=8, fontweight='bold', color=txt_col)
            y_pos += get_height(obj)
        current_ship_x += max_w + padding

    # CRANE CLAW FIELD
    if world.holding:
        obj = world.objects[world.holding]
        hold_x, hold_y = 11.0, 6.2
        ax.add_patch(draw_object(obj, hold_x, hold_y))
        # Dynamic text color adjustment inside crane box if applicable
        txt_col = "white" if obj.color in ["blue", "red", "green"] else "black"
        ax.text(hold_x, hold_y + get_height(obj)/2, f"{world.holding}\n({obj.weight}t)", ha='center', va='center', fontsize=8, fontweight='bold', color=txt_col)
        ax.text(hold_x, hold_y - 0.4, f"CRANE HOLDING", ha='center', fontsize=10, fontweight='bold', color='red')

    plt.tight_layout()
    buf = BytesIO()
    plt.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    buf.seek(0)
    img = Image.open(buf)
    plt.close(fig)
    return img

# ----------------------------------------
# CORE PROCESSING PIPELINES
# ----------------------------------------

def init_interface_data():
    global world_state
    world_state = create_initial_world()
    welcome_text = (
        "🏗️ **Welcome to the Port Logistics Terminal Simulator!**\n"
        "I am ready to help you move freight containers safely inside structural limits."
    )
    startup_history = [
        {"role": "assistant", "content": f"{welcome_text}\n\nHIII"}
    ]
    return startup_history, visualize_world(world_state), world_state.describe()

def process_command(command: str, chat_history):
    global world_state
    if not command or not command.strip():
        return chat_history, visualize_world(world_state), world_state.describe()

    if chat_history is None:
        chat_history = []
    chat_history.append({"role": "user", "content": command})

    try:
        response = interpret_and_act(world_state, command)
    except Exception as exc:
        response = f"❌ {str(exc)}"

    chat_history.append({"role": "assistant", "content": response})
    return chat_history, visualize_world(world_state), world_state.describe()

# ----------------------------------------
# Gradio UX Interface Compilation
# ----------------------------------------
with gr.Blocks(title="Cargo Ship Loading Simulator") as demo:
    gr.Markdown("# Cargo Ship Loading Simulator")
    gr.Markdown("Manage freight containers across three areas: **'loading area a'**, **'loading area b'**, and **'cargo ship'**.<br>"
    "<br>The available attributes are:<br>" \
    "• **Sizes:** {small, medium, large}<br>" \
    "• **Colors:** {red, green, blue, yellow, pink, orange}<br>" \
    "• **Shapes:** {triangle, square, rectangle, pentagon, circle}<br>"
    "<br>The available commands are:<br>" \
    "• **pick up X**<br>" \
    "• **put X on Y** or **stack X on Y**<br>" \
    "• **move X to Z** or **put X in Z**<br>" \
    "• **put down X** or just **put down**<br>" \
    "• **inspect X** or **view details of X**<br>"
    "*(Where X is one object, Y is another object, and Z is an area.)*")
    
    with gr.Row():
        with gr.Column(scale=1):
            chatbot = gr.Chatbot(label="Conversation History", height=650, show_label=True)
            with gr.Row():
                command_input = gr.Textbox(label="Enter Action Command", placeholder="e.g., view details of c1, inspect blue square, put down", scale=4)
                submit_btn = gr.Button("Execute", variant="primary", scale=1)
            with gr.Row():
                reset_btn = gr.Button("🔄 Reset Harbor Layout", variant="secondary")
            world_text = gr.Textbox(label="Data State Profile (Text Log)", lines=7, interactive=False)
        with gr.Column(scale=1):
            world_viz = gr.Image(label="Live Graphical Workspace Rendering", type="pil", height=800)
    
    submit_btn.click(fn=process_command, inputs=[command_input, chatbot], outputs=[chatbot, world_viz, world_text]).then(lambda: "", outputs=[command_input])
    command_input.submit(fn=process_command, inputs=[command_input, chatbot], outputs=[chatbot, world_viz, world_text]).then(lambda: "", outputs=[command_input])
    reset_btn.click(fn=init_interface_data, outputs=[chatbot, world_viz, world_text])
    demo.load(fn=init_interface_data, outputs=[chatbot, world_viz, world_text])

if __name__ == "__main__":
    demo.launch(share=False, server_name="0.0.0.0", height=800)

ERROR:    [Errno 10048] error while attempting to bind on address ('0.0.0.0', 7860): [winerror 10048] only one usage of each socket address (protocol/network address/port) is normally permitted
ERROR:    [Errno 10048] error while attempting to bind on address ('0.0.0.0', 7861): [winerror 10048] only one usage of each socket address (protocol/network address/port) is normally permitted


* Running on local URL:  http://0.0.0.0:7862
* To create a public link, set `share=True` in `launch()`.


c:\Users\kyria\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\kyria\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\kyria\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\kyria\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. 